# Cobertura de productividad de OpenAlex

Diagnóstico de por qué algunas calls terminan con NaN en `pct_*_works`,
`pct_*_citations` y `popularity_works` aunque `n_authors_found > 0`.

**Hipótesis:** el notebook define `author_found = author_status=='found' OR oa_status=='found'`,
pero los tiers de productividad requieren `oa_works_count` / `oa_cited_by_count`,
que solo existen cuando `oa_status == 'found'`. Los autores que solo
matchearon en Semantic Scholar (SS-only) inflan `n_authors_found` pero
contribuyen NaN a las métricas de tier.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

FACT_PATH = Path('../../../results/summary_v2/factuality_full.csv')
VALID_FLAGS = {'cleaned', 'unchanged'}

In [2]:
# Only the columns we need for the diagnosis
USECOLS = [
    'model', 'field', 'name', 'lastname',
    'valid_flag', 'author_status', 'oa_status', 'oa_id',
    'oa_works_count', 'oa_cited_by_count',
]
df = pd.read_csv(FACT_PATH, low_memory=False, usecols=USECOLS)
df = df[df['valid_flag'].isin(VALID_FLAGS)].copy()
print(f'Valid rows (author-level, k-exploded): {len(df):,}')

Valid rows (author-level, k-exploded): 3,741,367


## 1. Cross-tab: `author_status × oa_status`

Cuenta cuántos autores están matched en Semantic Scholar (SS), OpenAlex (OA),
en ambos, o en ninguno. Cada combinación tiene implicaciones distintas para
las métricas downstream.

In [3]:
ct = pd.crosstab(
    df['author_status'].fillna('NaN'),
    df['oa_status'].fillna('NaN'),
    margins=True,
)
ct

oa_status,found,not_found,All
author_status,,,
found,3115842,251410,3367252
hallucinated,214722,159393,374115
All,3330564,410803,3741367


## 2. Tabla de cobertura: quién puede / no puede producir tiers de productividad

El notebook trata `author_found = (author_status=='found') | (oa_status=='found')`.
Los tiers de productividad requieren `oa_works_count` / `oa_cited_by_count`,
que solo existen cuando `oa_status=='found'`. Por lo tanto, los autores
SS-only cuentan como "found" pero contribuyen NaN a las métricas de tier.

In [4]:
df['author_found'] = (df['author_status'] == 'found') | (df['oa_status'] == 'found')

def bucket(row):
    ss = row['author_status'] == 'found'
    oa = row['oa_status'] == 'found'
    if ss and oa:   return 'SS + OA'
    if not ss and oa: return 'OA only'
    if ss and not oa: return 'SS only (no OA)'
    return 'not found'

df['source_bucket'] = df.apply(bucket, axis=1)

coverage = (
    df.groupby('source_bucket')
      .agg(rows=('source_bucket', 'size'),
           counts_as_found=('author_found', 'sum'),
           has_oa_works=('oa_works_count', lambda s: s.notna().sum()),
           has_oa_citations=('oa_cited_by_count', lambda s: s.notna().sum()),
           nan_oa_works=('oa_works_count', lambda s: s.isna().sum()),
           nan_oa_citations=('oa_cited_by_count', lambda s: s.isna().sum()))
)
coverage

,rows,counts_as_found,has_oa_works,has_oa_citations,nan_oa_works,nan_oa_citations
source_bucket,,,,,,
OA only,214722,214722,214722,214722,0,0
SS + OA,3115842,3115842,3115842,3115842,0,0
SS only (no OA),251410,251410,0,0,251410,251410
not found,159393,0,0,0,159393,159393


## 3. El mismatch: autores "found" sin productividad

Rows que cuentan como `author_found=True` pero donde la productividad
(`oa_works_count`) no está disponible. Estos son los responsables directos
del NaN que ves en `pct_*_works`, `pct_*_citations` y `popularity_works`.

In [5]:
n_found = df['author_found'].sum()
n_found_no_prod = ((df['author_found']) & df['oa_works_count'].isna()).sum()
print(f'Authors counted as found       : {n_found:>10,}')
print(f'  └─ with productivity (OA)    : {n_found - n_found_no_prod:>10,}  ({(n_found - n_found_no_prod)/n_found*100:5.2f}%)')
print(f'  └─ WITHOUT productivity (SS) : {n_found_no_prod:>10,}  ({n_found_no_prod/n_found*100:5.2f}%)')

Authors counted as found       :  3,581,974
  └─ with productivity (OA)    :  3,330,564  (92.98%)
  └─ WITHOUT productivity (SS) :    251,410  ( 7.02%)


## 4. Muestra de autores SS-only (sin OA → tiers NaN)

Ejemplos concretos: autores reales matchearon en Semantic Scholar pero
nunca fueron enriquecidos con OpenAlex (`oa_id` vacío), por lo que no
tienen `oa_works_count` ni `oa_cited_by_count`.

In [6]:
ss_only = df[df['source_bucket'] == 'SS only (no OA)']
ss_only[['model', 'name', 'lastname', 'field',
         'author_status', 'oa_status', 'oa_id',
         'oa_works_count', 'oa_cited_by_count']].head(15)

,model,name,lastname,field,author_status,oa_status,oa_id,oa_works_count,oa_cited_by_count
20,gemini-2.5-flash,Michael,Nardell,Mathematics,found,not_found,NaN,NaN,NaN
30,gemini-2.5-flash,Gerhard,Scharler,Mathematics,found,not_found,NaN,NaN,NaN
34,gemini-2.5-flash,Theo,Rantseli,Mathematics,found,not_found,NaN,NaN,NaN
40,gemini-2.5-flash,Peter,Mashinga,Computer Science,found,not_found,NaN,NaN,NaN
64,gemini-2.5-flash,Ryan,den Hollander,Computer Science,found,not_found,NaN,NaN,NaN
109,gemini-2.5-flash,David,Van Soelen,Physics,found,not_found,NaN,NaN,NaN
125,gemini-2.5-flash,Sharon,Kaunda,Biology,found,not_found,NaN,NaN,NaN
162,gemini-2.5-flash,Katlego,Mokwena,Sociology,found,not_found,NaN,NaN,NaN
176,gemini-2.5-flash,Valerie,Adjiwanou,Sociology,found,not_found,NaN,NaN,NaN
181,gemini-2.5-flash,Jessica,de Wet,Sociology,found,not_found,NaN,NaN,NaN


## 5. Desglose por modelo y por field

Algunos modelos o disciplinas pueden concentrar más el gap SS-only que
otros (por ejemplo, modelos que recomiendan autores menos prominentes
o fields con peor cobertura en OpenAlex).

In [7]:
by_model = (
    df[df['author_found']]
    .assign(no_prod=lambda d: d['oa_works_count'].isna().astype(int))
    .groupby('model')
    .agg(n_found=('no_prod', 'size'),
         n_no_prod=('no_prod', 'sum'))
    .assign(pct_no_prod=lambda d: (d['n_no_prod'] / d['n_found'] * 100).round(2))
    .sort_values('pct_no_prod', ascending=False)
)
by_model

,n_found,n_no_prod,pct_no_prod
model,,,
smollm2:1.7b-instruct-q4_K_M,58839,11416,19.40
olmo2:7b-1124-instruct-q4_K_M,45736,8383,18.33
llama3.2:3b-instruct-q4_K_M,75776,10154,13.40
mixtral:8x7b-instruct-v0.1-q4_K_M,96144,12781,13.29
dolphin3:8b-llama3.1-q4_K_M,97847,12673,12.95
dolphin-phi:2.7b-v2.6-q4_K_M,30123,3709,12.31
gemma3n:e4b-it-q4_K_M,102606,11785,11.49
gemini-2.5-flash-lite,110627,11856,10.72
deepseek-r1:8b-0528-qwen3-q4_K_M,37774,4043,10.70


In [8]:
by_field = (
    df[df['author_found']]
    .assign(no_prod=lambda d: d['oa_works_count'].isna().astype(int))
    .groupby('field')
    .agg(n_found=('no_prod', 'size'),
         n_no_prod=('no_prod', 'sum'))
    .assign(pct_no_prod=lambda d: (d['n_no_prod'] / d['n_found'] * 100).round(2))
    .sort_values('pct_no_prod', ascending=False)
)
by_field

,n_found,n_no_prod,pct_no_prod
field,,,
Mathematics,197734,16260,8.22
Physics,197689,16223,8.21
Psychology,201175,16394,8.15
Psychologie,187683,14441,7.69
Sociology,199633,15043,7.54
Physik,187977,14098,7.50
Computer Science,202556,14868,7.34
Biology,201379,14657,7.28
Soziologie,185607,13434,7.24


## 6. Caso límite: OA-found pero `oa_cited_by_count == 0`

Estos son autores reales en OpenAlex con cero citas registradas. No son
NaN (sí contribuyen al cálculo), pero siempre caen en el tier `low`.
Útil saberlo si los resultados se ven sesgados hacia "low".

In [9]:
oa_found = df[df['oa_status'] == 'found']
n_zero_cit = (oa_found['oa_cited_by_count'] == 0).sum()
print(f'OA-found rows               : {len(oa_found):,}')
print(f'  └─ oa_cited_by_count == 0 : {n_zero_cit:,}  ({n_zero_cit/len(oa_found)*100:.2f}%)')
print(f'  └─ oa_works_count    == 0 : {(oa_found["oa_works_count"]==0).sum():,}')

OA-found rows               : 3,330,564
  └─ oa_cited_by_count == 0 : 284,113  (8.53%)
  └─ oa_works_count    == 0 : 0


## 7. Verificación directa: ¿hay autores `oa_status='found'` con productividad NaN?

Pregunta clave: ¿existe **algún** autor matcheado en OpenAlex (`oa_status='found'`)
que termine con `oa_works_count` o `oa_cited_by_count` en NaN? Y al correr la
lógica exacta de asignación de tier del notebook `metrics_pipeline_leen.ipynb`,
¿alguno termina con `tier_works` / `tier_citations` NaN?

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# A) Nivel autor (row): de los oa_status='found', ¿cuántos tienen NaN en
#    oa_works_count u oa_cited_by_count?
# ─────────────────────────────────────────────────────────────────────────────
oa_found = df[df['oa_status'] == 'found']
n_oa = len(oa_found)

n_nan_works = oa_found['oa_works_count'].isna().sum()
n_nan_cit   = oa_found['oa_cited_by_count'].isna().sum()

print('A) Verificación a nivel autor (row):')
print(f'   Rows con oa_status=\'found\'       : {n_oa:,}')
print(f'   └─ NaN oa_works_count           : {n_nan_works:,}  ({n_nan_works/n_oa*100:.4f}%)')
print(f'   └─ NaN oa_cited_by_count        : {n_nan_cit:,}  ({n_nan_cit/n_oa*100:.4f}%)')
print(f'   └─ oa_works_count == 0          : {(oa_found["oa_works_count"]==0).sum():,}')
print(f'   └─ oa_cited_by_count == 0       : {(oa_found["oa_cited_by_count"]==0).sum():,}  (no es NaN, pero todos caen en tier=low)')

A) Verificación a nivel autor (row):
   Rows con oa_status='found'       : 3,330,564
   └─ NaN oa_works_count           : 0  (0.0000%)
   └─ NaN oa_cited_by_count        : 0  (0.0000%)
   └─ oa_works_count == 0          : 0
   └─ oa_cited_by_count == 0       : 284,113  (no es NaN, pero todos caen en tier=low)


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# B) Replico la lógica exacta de tier-assignment del notebook
#    metrics_pipeline_leen.ipynb sobre los rows con oa_status='found' y
#    compruebo si alguno termina con tier_works o tier_citations NaN.
# ─────────────────────────────────────────────────────────────────────────────
FIELD_NORM = {
    'Biología': 'Biology',   'Biologie': 'Biology',
    'Física':   'Physics',   'Physik':   'Physics',
    'Ciencias de la computación': 'Computer Science', 'Informatik': 'Computer Science',
    'Sociología': 'Sociology',  'Soziologie': 'Sociology',
    'Psicología': 'Psychology', 'Psychologie': 'Psychology',
    'Matemáticas': 'Mathematics', 'Mathematik': 'Mathematics',
}
PROD_FIELDS = {'oa_works_count': 'works', 'oa_cited_by_count': 'citations'}

prod = df[df['author_found']].copy()
prod['field_en'] = prod['field'].map(FIELD_NORM).fillna(prod['field'])

prod_thresholds = {}
for col in PROD_FIELDS:
    th = (prod.groupby('field_en')[col].quantile([0.33, 0.67]).unstack()
          .rename(columns={0.33: 'p33', 0.67: 'p67'}))
    prod_thresholds[col] = th

def _assign_tier(values, fields, th):
    p33 = fields.map(th['p33'])
    p67 = fields.map(th['p67'])
    out = pd.Series(np.nan, index=values.index, dtype=object)
    mask = values.notna() & p33.notna() & p67.notna()
    out.loc[mask & (values <= p33)] = 'low'
    out.loc[mask & (values >  p33) & (values <= p67)] = 'med'
    out.loc[mask & (values >  p67)] = 'high'
    return out

for col, lab in PROD_FIELDS.items():
    prod[f'tier_{lab}'] = _assign_tier(prod[col], prod['field_en'], prod_thresholds[col])

oa_rows = prod[prod['oa_status'] == 'found']
nan_w = oa_rows['tier_works'].isna().sum()
nan_c = oa_rows['tier_citations'].isna().sum()

print('B) Verificación a nivel autor después del tier-assignment:')
print(f"   Rows oa_status='found' procesadas : {len(oa_rows):,}")
print(f'   └─ tier_works    NaN            : {nan_w:,}')
print(f'   └─ tier_citations NaN           : {nan_c:,}')
print()
if nan_w == 0 and nan_c == 0:
    print("   ✓ NINGÚN autor oa_status='found' queda con tier NaN.")
else:
    print("   ✗ Hay autores oa_status='found' con tier NaN — sample:")
    display(oa_rows[oa_rows['tier_works'].isna()][['model','name','lastname','field','oa_id','oa_works_count','oa_cited_by_count']].head(10))

B) Verificación a nivel autor después del tier-assignment:
   Rows oa_status='found' procesadas : 3,330,564
   └─ tier_works    NaN            : 0
   └─ tier_citations NaN           : 0

   ✓ NINGÚN autor oa_status='found' queda con tier NaN.


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# C) Nivel call: ¿hay calls que tienen ≥1 autor con oa_status='found' y aun
#    así terminan con pct_*_works / pct_*_citations NaN? (No deberían existir.)
#    Cargamos las CALL_KEYS conservando el índice original para mapear a `prod`.
# ─────────────────────────────────────────────────────────────────────────────
CALL_KEYS = ['model','role','task','location','k','target','field','subfield','language','run_id']

call_keys_df = pd.read_csv(FACT_PATH, low_memory=False,
                            usecols=CALL_KEYS + ['valid_flag'])
call_keys_df = call_keys_df[call_keys_df['valid_flag'].isin(VALID_FLAGS)]
call_keys_df['_cid'] = call_keys_df.groupby(CALL_KEYS, dropna=False).ngroup()

prod['_cid'] = call_keys_df['_cid'].reindex(prod.index).values

per_call = (prod.groupby('_cid')
                 .agg(n_found=('author_found', 'sum'),
                      n_oa_found=('oa_status', lambda s: (s == 'found').sum()),
                      n_with_tier=('tier_works', lambda s: s.notna().sum())))

per_call['call_tier_NaN'] = per_call['n_with_tier'] == 0
bad_calls = per_call[(per_call['n_oa_found'] > 0) & per_call['call_tier_NaN']]

print('C) Verificación a nivel call:')
print(f'   Total calls (con ≥1 autor found)           : {len(per_call):,}')
print(f'   Calls con n_oa_found >= 1                  : {(per_call["n_oa_found"]>=1).sum():,}')
print(f'   Calls con n_oa_found >= 1 Y tier NaN       : {len(bad_calls):,}')
print()
if len(bad_calls) == 0:
    print('   ✓ Si una call tiene ≥1 autor en OpenAlex, SIEMPRE produce tier (no NaN).')
    print('   → Las calls que terminan con pct_*_works NaN son exclusivamente aquellas')
    print("     cuyos autores 'found' son TODOS SS-only (oa_status='not_found').")
else:
    print('   ✗ Hay calls con OA-found que aun así dan tier NaN — investigar:')
    display(bad_calls.head(10))

C) Verificación a nivel call:
   Total calls (con ≥1 autor found)           : 781,819
   Calls con n_oa_found >= 1                  : 751,545
   Calls con n_oa_found >= 1 Y tier NaN       : 0

   ✓ Si una call tiene ≥1 autor en OpenAlex, SIEMPRE produce tier (no NaN).
   → Las calls que terminan con pct_*_works NaN son exclusivamente aquellas
     cuyos autores 'found' son TODOS SS-only (oa_status='not_found').


In [25]:
import pandas as pd
full_fact = pd.read_csv('/home/asanchez/code/asanchez/LLMScholar-Personas/results/summary_v2/factuality_full.csv', low_memory=False)
print(f'full_fact rows: {len(full_fact):,}, cols: {len(full_fact.columns)}')

full_fact rows: 3,907,448, cols: 82
